# W9 Lab — A RAG Program for the GDPR

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week09/W9_lab_gdpr_rag.ipynb)

**Goal.** Practice 2 built retrieval-augmented generation (RAG) for the EU AI Act one step at a time. This lab puts the same steps into one program of three functions and runs it on a second regulation, the EU General Data Protection Regulation (**GDPR**): 99 articles, about 30,000 words. It then measures a choice that Practice 2 kept fixed, the **chunk size**. At the end, the same program runs on another document.

| Part | Question |
|---|---|
| 1. The program | Which code does a RAG program need? |
| 2. The GDPR in a vector store | How many sections and chunks does the GDPR give? |
| 3. Questions and answers | Do the answers name their articles, and does the program say when the GDPR has no answer? |
| 4. Hit rate | How often does the search find the right article? |
| 5. Chunk size | How does the chunk size change the hit rate and the length of the input? |
| 6. Another document | How does the program run on a document of your own? |

**Cell labels.** The first line of each code cell says what to do with it:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.

It takes about 50 minutes. Do Practice 2 (RAG over the EU AI Act) first: this lab uses its parts without explaining them again. The text of the GDPR comes from EUR-Lex, without the recitals and the footnotes; only the version in the Official Journal of the EU is legally binding.

## 0. Setup

*Do:* run the three cells; paste your API key into the second. The third cell makes the chat model and the embedding model, as in Practice 2, and downloads the text of the GDPR into the file `gdpr.md`.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14" "langchain-text-splitters==1.1.3"

In [ ]:
# ▶ RUN · setup: just run
import os

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

In [ ]:
# ▶ RUN · setup: just run
import urllib.request
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

model = init_chat_model("openai:gpt-4.1")                            # the chat model: writes the answers
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")    # the embedding model: turns texts into vectors

DATA_URL = "https://raw.githubusercontent.com/ralbu85/stml_2026/main/labs/data/"
DOC_PATH = "gdpr.md"
if not os.path.exists(DOC_PATH):   # download the text once
    urllib.request.urlretrieve(DATA_URL + DOC_PATH, DOC_PATH)
gdpr_text = open(DOC_PATH, encoding="utf-8").read()
print(len(gdpr_text.split()), "words")

## 1. The program

A RAG program has two stages: it prepares the document once, and then it answers each question from the passages that it retrieves. Three functions hold all the steps of Practice 2:

| Function | What it does | Practice 2 |
|---|---|---|
| `split_document(text, chunk_size)` | cuts the text at the `#` and `##` headings, then cuts each section into chunks of at most `chunk_size` characters | Part 2 |
| `build_store(chunks)` | embeds every chunk and keeps it in a vector store | Part 3 |
| `rag_answer(store, question, k)` | retrieves the *k* most similar chunks, puts them into the user message under their section labels, and answers under the three grounding rules | Parts 4 and 5 |

Two things are new. The chunk size is a parameter, so Part 5 can change it. The store is a parameter of `rag_answer`, so one program can answer from several stores: Part 5 builds three and Part 6 one more. The grounding rules name no document, so they hold for any document.

### 1.1 Preparing a document

*Do:* read the two functions, then run the cell. It only defines them; Part 2 runs them. The overlap is one tenth of the chunk size (`//` divides and drops the remainder).

In [ ]:
# ▶ RUN · example: read each line, then run
def split_document(text, chunk_size):
    """Cut the text at the # and ## headings, then cut each section into chunks of at most chunk_size characters."""
    header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "chapter"), ("##", "section")])
    sections = []
    for piece in header_splitter.split_text(text):
        if "section" in piece.metadata:            # leave out the text before the first ## heading
            sections.append(piece)
    size_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_size // 10)
    return size_splitter.split_documents(sections)   # each chunk keeps the labels of its section

def build_store(chunks):
    """Embed every chunk and keep each embedding with its text and metadata."""
    store = InMemoryVectorStore(embedding_model)
    store.add_documents(chunks)
    return store

### 1.2 Answering a question

*Do:* read the rules and the three functions, then run the cell. `format_passages` and `build_messages` are the functions of Practice 2. `rag_answer` returns the answer and the retrieved chunks, so that Part 3 can print the sections next to the answer.

In [ ]:
# ▶ RUN · example: read each line, then run
RAG_SYSTEM = """You answer questions about a document.
1. Answer only from the passages in the user message. Do not add facts from your own knowledge.
2. After each fact, name its section, for example (Article 33(1)).
3. If the passages do not contain the answer, say so and name what is missing."""

def format_passages(docs):
    """Write each chunk under its section label, for example "[Article 33: Notification of ...]"."""
    parts = []
    for doc in docs:
        parts.append("[" + doc.metadata["section"] + "]\n" + doc.page_content)
    return "\n\n".join(parts)

def build_messages(passages, question):
    """The system message holds the rules; the user message holds the passages and the question."""
    return [SystemMessage(RAG_SYSTEM), HumanMessage("Passages:\n" + passages + "\n\nQuestion: " + question)]

def rag_answer(store, question, k=3):
    """Retrieve k chunks from the store, put them into the input, and answer from them."""
    docs = store.as_retriever(search_kwargs={"k": k}).invoke(question)   # 1. retrieve
    messages = build_messages(format_passages(docs), question)           # 2. put the passages into the input
    reply = model.invoke(messages)                                       # 3. answer
    return reply.content, docs

## 2. The GDPR in a vector store

*Do:* run the cell. It splits the GDPR into chunks of at most 1,500 characters, the size of Practice 2, and embeds them. Then it prints the number of sections and chunks, and the label and the beginning of one chunk.

In [ ]:
# ▶ RUN · example: read each line, then run
gdpr_chunks = split_document(gdpr_text, 1500)
gdpr_store = build_store(gdpr_chunks)

section_names = []
for chunk in gdpr_chunks:
    if chunk.metadata["section"] not in section_names:
        section_names.append(chunk.metadata["section"])
print(len(section_names), "sections,", len(gdpr_chunks), "chunks")
print(gdpr_chunks[60].metadata["section"])
print(gdpr_chunks[60].page_content[:300])

**Reading the output.** The 99 sections are the 99 articles of the GDPR; it has no annexes. 42 articles are longer than 1,500 characters, so the split makes 186 chunks. The printed chunk is Article 31, a short article that fits in one chunk. The same two functions prepared the EU AI Act in Practice 2 (126 sections, 298 chunks).

## 3. Questions and answers

*Do:* run the cell. It asks four questions. For each, it prints the sections of the three passages and the answer. Check the three rules: does each fact name an article (rule 2), is that article among the printed sections (rule 1), and does the answer say so when the sections do not hold the answer (rule 3)? The last question is the chatbot question of Practice 2, Part 3. The GDPR has no article about it.

In [ ]:
# ▶ RUN · example: read each line, then run
QUESTIONS = [
    "How soon must a controller notify the supervisory authority of a personal data breach?",
    "What is the maximum fine for infringing the basic principles for processing?",
    "Can a person ask a company to delete their personal data?",
    "Must a chatbot tell people that it is an AI?",
]
for question in QUESTIONS:
    answer, docs = rag_answer(gdpr_store, question)
    print("QUESTION:", question)
    print("SECTIONS:")
    for doc in docs:
        print("   ", doc.metadata["section"])
    print("ANSWER:  ", answer)
    print("-" * 80)

**Reading the output.** In our runs, the first three answers named the article of each fact, and that article was among the printed sections: Article 33(1) for the 72 hours, Article 83(5) for EUR 20 000 000 or 4 % of the total worldwide annual turnover, and Article 17(1) for the right to erasure. The third question says "delete" and the GDPR says "erasure", but the search still found Article 17. For the chatbot question, the passages came from Articles 32, 12, and 22, and the answer said that they do not contain such a duty, as rule 3 asks. The search always returns *k* chunks, even when the document has no answer; the rules decide what the model does with them. Part 6 asks the same question about the EU AI Act.

## 4. Hit rate

A question is a **hit** when the article that answers it is among the *k* retrieved chunks, and a **miss** when it is not. The **hit rate at k** (hit@k) is the share of hits in a set of questions, as in Practice 2, Part 7. `EVAL_SET` holds twelve questions about the GDPR, each with the article that answers it, found by reading the regulation. Most questions use everyday words, not the words of the article: "delete" instead of "erasure", "answer a request" instead of "provide information on action taken".

`hit_rate(store, k)` returns the number of hits. With `show=True` it also prints one line for each question.

*Do:* run the cell.

In [ ]:
# ▶ RUN · example: read each line, then run
EVAL_SET = [
    ("When must a company tell the affected people themselves about a personal data breach?", "Article 34"),
    ("Can a person ask a company to delete their personal data?", "Article 17"),
    ("Can a person get their data in a machine-readable format and take it to another company?", "Article 20"),
    ("When must a company appoint a data protection officer?", "Article 37"),
    ("When is a data protection impact assessment required?", "Article 35"),
    ("From what age can a child consent to an online service?", "Article 8"),
    ("On which legal grounds may personal data be processed?", "Article 6"),
    ("What is the maximum fine for infringing the basic principles for processing?", "Article 83"),
    ("Does the GDPR apply to a company outside the EU that sells to people in the EU?", "Article 3"),
    ("May a company process health data?", "Article 9"),
    ("How long does a company have to answer a request from a data subject?", "Article 12"),
    ("Can a person object to direct marketing?", "Article 21"),
]

def hit_rate(store, k, show=True):
    """Return the number of questions in EVAL_SET whose article is among the k chunks found."""
    hits = 0
    for question, expected in EVAL_SET:
        found = []
        for doc in store.as_retriever(search_kwargs={"k": k}).invoke(question):
            found.append(doc.metadata["section"].split(":")[0])   # "Article 17: Right to erasure ..." -> "Article 17"
        if expected in found:
            hits = hits + 1
            result = "hit "
        else:
            result = "MISS"
        if show:
            print(result, "| expected:", expected, "| found:", found)
    if show:
        print("hit@" + str(k) + ":", hits, "/", len(EVAL_SET))
    return hits

hit_rate(gdpr_store, 3)

**Reading the output.** In our runs, all twelve questions were hits: hit@3 was 12 of 12. Some questions got two or three chunks of the same article, for example Articles 35 and 83: a long article is split into several chunks, and each of them can be similar to the question. Part 5 changes the chunk size and measures again.

## 5. Chunk size

The chunk size decides how much text one embedding represents.

- **Small chunks:** each embedding stands for a short text about one point. Several chunks of one long article can fill the top *k* places, and the answer can be cut between two chunks.
- **Large chunks:** each embedding stands for a long text, often a whole article with several points. The *k* passages make a long input, and the model reads more text that the question does not need.

*Do:* run the cell. It builds one store for each of three chunk sizes: 500, 1,500, and 4,000 characters. Most articles of the GDPR are shorter than 4,000 characters, so that size keeps most articles whole. For each size, the cell prints the number of chunks, hit@3 on `EVAL_SET`, and the average length of the passages that `k=3` puts into the input, in characters. Embedding the three stores takes about half a minute.

In [ ]:
# ▶ RUN · example: read each line, then run
stores = {}
for size in [500, 1500, 4000]:
    chunks = split_document(gdpr_text, size)
    stores[size] = build_store(chunks)
    hits = hit_rate(stores[size], 3, show=False)
    total = 0
    for question, expected in EVAL_SET:
        docs = stores[size].as_retriever(search_kwargs={"k": 3}).invoke(question)
        total = total + len(format_passages(docs))
    print("chunk size", size, "|", len(chunks), "chunks | hit@3:", hits, "/", len(EVAL_SET),
          "| passages:", round(total / len(EVAL_SET)), "characters")

**Reading the output.** In our runs, the three sizes gave 528, 186, and 116 chunks. hit@3 was 12 of 12 with 500 and 1,500 characters, and 11 of 12 with 4,000. The passages of `k=3` had about 1,300, 3,700, and 5,900 characters: the larger the chunks, the longer the input for the same *k*. The miss at 4,000 was the question about the time limit for an answer to a data subject. The next cell shows why.

*Do:* run the cell below. It asks the question about the time limit for an answer to a data subject with each store and prints the answer. The answer is in Article 12(3): one month, which can be extended by two further months.

In [ ]:
# ▶ RUN · example: read each line, then run
question = "How long does a company have to answer a request from a data subject?"
for size in [500, 1500, 4000]:
    answer, docs = rag_answer(stores[size], question)
    print("CHUNK SIZE", size)
    for doc in docs:
        print("   ", doc.metadata["section"], "|", len(doc.page_content), "characters")
    print(answer)
    print("-" * 80)

**Reading the output.** In our runs, with 500 characters, two of the three chunks came from Article 12, and both had the words "one month". With 1,500 characters, the middle chunk of Article 12 held paragraph 3. Both answers gave one month, which can be extended by two further months (Article 12(3)). With 4,000 characters, Article 12 was one chunk of 3,492 characters with eight paragraphs: the form of the information, the exercise of the rights, the time limit, a refusal, fees, identity checks, icons, and delegated acts. One embedding stood for all eight points, and it was less similar to the question than the chunks of Articles 15, 16, and 33. Article 12 was not retrieved, and the answer said that the passages do not give the time limit (rule 3).

A chunk with many points can lose to a chunk about one point, and a small chunk carries less context and makes more chunks. For the GDPR, 1,500 characters found every article of `EVAL_SET`; for another document, measure hit@k with a few sizes before choosing one.

## 6. Another document

The program works for any Markdown file with `#` and `##` headings: the `##` headings become the section labels that the answers name. The cell below runs it on the EU AI Act of Practice 2 and asks the chatbot question of Part 3 again.

*Do:* run the cell. Then try a document of your own: upload a Markdown file in the **Files** panel of Colab (the folder icon on the left), write its file name in `MY_PATH` and a question about it in `MY_QUESTION`, and run the cell again. A text without `##` headings gives no sections; add a `##` heading before each part first.

In [ ]:
# ▶ RUN · example: read each line, then run
MY_PATH = "eu_ai_act.md"                                     # change this to the file name of your document
MY_QUESTION = "Must a chatbot tell people that it is an AI?"   # change this to a question about your document

if not os.path.exists(MY_PATH):                              # the EU AI Act is downloaded; your file is uploaded
    urllib.request.urlretrieve(DATA_URL + MY_PATH, MY_PATH)
my_store = build_store(split_document(open(MY_PATH, encoding="utf-8").read(), 1500))
answer, docs = rag_answer(my_store, MY_QUESTION)
print("SECTIONS:")
for doc in docs:
    print("   ", doc.metadata["section"])
print("ANSWER:  ", answer)

**Reading the output.** In our runs, all three passages came from Article 50 of the EU AI Act, and the answer said yes: a chatbot must inform people that they are interacting with an AI system, unless this is obvious from the circumstances (Article 50(1)). From the GDPR, the same question got "the passages do not say" in Part 3. The program was the same; only the document changed. For a document of your own, check the answers in the same way: the printed sections, the article or section named after each fact, and the answer to a question that the document does not cover.